In [1]:
import uproot
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D

import h5py as h5
import math
import os

import pandas as pd
import plotly.graph_objects as go

import plotly.express as px
import sys

from ipywidgets import interact

In [2]:
SOFTWARE_DIR = '/global/homes/k/kjvocker/spine/src' # Path to software install
DATA_DIR = '/global/homes/k/kjvocker/data/'
Mx2_DIR = "/global/cfs/cdirs/dune/www/data/2x2/simulation/productions/MiniRun5_1E19_RHC/MiniRun5_1E19_RHC.minerva/DST/0000000/"
SPINE_DIR = "/global/cfs/cdirs/dune/www/data/2x2/simulation/productions/MiniRun6.4_1E19_RHC/MiniRun6.4_1E19_RHC.spine_v2/MLRECO_SPINE/0000000/"

TMS_DIR = "/global/cfs/cdirs/dunepro/people/abooth/nd-production/output/MiniProdN5/run-tms-reco/MiniProdN5p1_NDComplex_FHC.tmsreco.full.sanddrift/TMSRECO/0002000/"
NDLAr_DIR = "/global/cfs/cdirs/dunepro/people/abooth/nd-production/output/MiniProdN5/run-mlreco/MiniProdN5p1_NDComplex_FHC.spine.full.sanddrift/MLRECO_SPINE/0002000/"

filenum = 471

TMS_file = f"{TMS_DIR}MiniProdN5p1_NDComplex_FHC.tmsreco.full.sanddrift.0002{filenum}.TMSRECO.root"
SPINE_file = f"{NDLAr_DIR}MiniProdN5p1_NDComplex_FHC.spine.full.sanddrift.0002{filenum}.MLRECO_SPINE.hdf5"

# Set software directory
sys.path.append(SOFTWARE_DIR)
import yaml
from spine.driver import Driver

Welcome to JupyROOT 6.26/16


## Colour Palette

In [3]:
NDLAr_colour = '#e63946'

# List 1: Accessible Base (Strong & Distinct)
strong_colours = [
    '#E69F00', # Orange
    '#0072B2', # Deep Blue
    '#009E73', # Bluish Green
    '#CC79A7', # Reddish Purple
    '#D55E00', # Vermillion
    '#56B4E9', # Sky Blue
    '#F0E442'  # Yellow
]

# List 2: Accessible Tint (Lighter "Shadow" Versions)
light_colours = [
    '#f0c975', # Light Orange
    '#70b1d9', # Pale Blue
    '#7ecdb9', # Light Green
    '#e0adc9', # Light Orchid
    '#e8a176', # Light Salmon
    '#a2d9f7', # Very Light Blue
    '#f7f2a1'  # Pale Yellow
]

## Create ND-LAr and TMS Geometry

In [4]:
# Geometry defined in TMS_Constants.h:

#  // Approximate starting and end positions of TMS detector in geometry for plotting hits, in {x,y,z}
#  // in mm by default!
#  const double TMS_Start[] = {-4000, -3500, 11000};
#  const double TMS_End[] = {4000, 500, 19000};

# const double TMS_Double_End = 18502.5;  //this is still with problems in the geometry!!!
# const double TMS_Thin_Start = 11177.5;  //start of TMS at 1116 cm - bar thickness / 2 (8 mm) = 11160 - 8 = 11152 mm
# const double TMS_Start_Exact[] = {-3730, -3702.23, TMS_Thin_Start};
# const double TMS_End_Exact[] = {3730, 997.77, TMS_Double_End};

#  // From scanning all hits x,y,z position in the LAr active volume
#  const double LAr_Start_Exact[] = {-3478.48, -2166.71, 4179.24};
#  const double LAr_End_Exact[] = {3478.48, 829.282, 9135.88};

In [5]:
def plot_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, color, name):
    # This path visits every corner of a box in one continuous line.
    # We use 'None' to jump between disconnected edges.
    x = [x_min, x_max, x_max, x_min, x_min, x_min, x_max, x_max, x_min, x_min, None, x_max, x_max, None, x_min, x_min, None, x_max, x_max]
    y = [y_min, y_min, y_max, y_max, y_min, y_min, y_min, y_max, y_max, y_min, None, y_min, y_min, None, y_max, y_max, None, y_max, y_max]
    z = [z_min, z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max, z_max, None, z_min, z_max, None, z_min, z_max, None, z_min, z_max]

    fig.add_trace(
        go.Scatter3d(
            x=x, y=y, z=z,
            mode="lines",
            line=dict(color=color, width=3),
            name=name,
            showlegend=True
        )
    )

## Class to pull data from TMS file

In [6]:
class TMSData:
    def __init__(self, filename):
        self.file = uproot.open(filename)

        # Truth Info
        self.EventNo = self.file["Truth_Info"]["EventNo"].array(library="np")
        self.SpillNo = self.file["Truth_Info"]["SpillNo"].array(library="np")
        self.RunNo = self.file["Truth_Info"]["RunNo"].array(library="np")
        self.Interation = self.file["Truth_Info"]["Interaction"].array(library="np")
        
        self.TruthInfoIndex = self.file["Truth_Info"]["TruthInfoIndex"].array(library="np")
        self.TruthInfoNSlices = self.file["Truth_Info"]["TruthInfoNSlices"].array(library="np")
        self.nPrimaryVertices = self.file["Truth_Info"]["nPrimaryVertices"].array(library="np")
        self.nParticles = self.file["Truth_Info"]["nParticles"].array(library="np")
        self.MuonVertex = self.file["Truth_Info"]["Muon_Vertex"].array(library="np")
        self.VertexIdOfMostEnergyInEvent = self.file["Truth_Info"]["VertexIdOfMostEnergyInEvent"].array(library="np")
        self.RecoTrackN = self.file["Truth_Info"]["RecoTrackN"].array(library="np")

        self.NeutrinoX4 = self.file["Truth_Info"]["NeutrinoX4"].array(library="np")

        self.RecoTrackPrimaryParticleIndex = self.file["Truth_Info"]["RecoTrackPrimaryParticleIndex"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionTrackStart = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionTrackStart"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionTrackEnd = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionTrackEnd"].array(library="np")
        self.RecoTrackTrueHitPosition = self.file["Truth_Info"]["RecoTrackTrueHitPosition"].array(library="np")
        self.RecoTrackPrimaryParticlePDG = self.file["Truth_Info"]["RecoTrackPrimaryParticlePDG"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionStart = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionStart"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionEnd = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionEnd"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionEnteringTMS = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionEnteringTMS"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionLeavingTMS = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionLeavingTMS"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionLeavingLAr = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionLeavingLAr"].array(library="np")
        self.RecoTrackPrimaryParticleVtxID = self.file["Truth_Info"]["RecoTrackPrimaryParticleVtxId"].array(library="np")

        self.RecoTrackSecondaryParticleIndex = self.file["Truth_Info"]["RecoTrackSecondaryParticleIndex"].array(library="np")
        self.RecoTrackSecondaryParticlePDG = self.file["Truth_Info"]["RecoTrackSecondaryParticlePDG"].array(library="np")
        self.RecoTrackSecondaryParticleTruePositionStart = self.file["Truth_Info"]["RecoTrackSecondaryParticleTruePositionStart"].array(library="np")
        self.RecoTrackSecondaryParticleTruePositionEnd = self.file["Truth_Info"]["RecoTrackSecondaryParticleTruePositionEnd"].array(library="np")
 
        self.TrueHitX = self.file["Truth_Info"]["TrueHitX"].array(library="np")
        self.TrueHitY = self.file["Truth_Info"]["TrueHitY"].array(library="np")
        self.TrueHitZ = self.file["Truth_Info"]["TrueHitZ"].array(library="np")
        self.TrueHitT = self.file["Truth_Info"]["TrueHitT"].array(library="np")
        self.TrueRecoHitX = self.file["Truth_Info"]["TrueRecoHitX"].array(library="np")
        self.TrueRecoHitY = self.file["Truth_Info"]["TrueRecoHitY"].array(library="np")
        self.TrueRecoHitZ = self.file["Truth_Info"]["TrueRecoHitZ"].array(library="np")
        self.TrueRecoHitT = self.file["Truth_Info"]["TrueRecoHitT"].array(library="np")

        self.TrueVtxN = self.file["Truth_Spill"]["TrueVtxN"].array(library="np")
        self.TrueVtxX = self.file["Truth_Spill"]["TrueVtxX"].array(library="np")
        self.TrueVtxY = self.file["Truth_Spill"]["TrueVtxY"].array(library="np")
        self.TrueVtxZ = self.file["Truth_Spill"]["TrueVtxZ"].array(library="np")
        self.TrueVtxT = self.file["Truth_Spill"]["TrueVtxT"].array(library="np")

        self.PDG = self.file["Truth_Spill"]["PDG"].array(library="np")
        self.TrueVtxPDG = self.file["Truth_Spill"]["TrueVtxPDG"].array(library="np")
        self.TrueVtxID = self.file["Truth_Spill"]["TrueVtxID"].array(library="np")
        self.VertexID = self.file["Truth_Spill"]["VertexID"].array(library="np")
        self.TrackID = self.file["Truth_Spill"]["TrackId"].array(library="np")
        self.TruthInfoIndex = self.file["Truth_Spill"]["TruthInfoIndex"].array(library="np")

        self.PositionLArStart = self.file["Truth_Spill"]["PositionLArStart"].array(library="np")
        self.PositionLArEnd = self.file["Truth_Spill"]["PositionLArEnd"].array(library="np")
        self.PositionTMSStart = self.file["Truth_Spill"]["PositionTMSStart"].array(library="np")
        self.PositionTMSEnd = self.file["Truth_Spill"]["PositionTMSEnd"].array(library="np")
        self.BirthPosition = self.file["Truth_Spill"]["BirthPosition"].array(library="np")
        self.DeathPosition = self.file["Truth_Spill"]["DeathPosition"].array(library="np")

        self.EventNo_TruthSpill = self.file["Truth_Spill"]["EventNo"].array(library="np")
        self.SpillNo_TruthSpill = self.file["Truth_Spill"]["SpillNo"].array(library="np")

        # Reco Tree
        self.RECO_StartPos = self.file["Reco_Tree"]["StartPos"].array(library="np")
        self.RECO_EndPos = self.file["Reco_Tree"]["EndPos"].array(library="np")

        # Line Candidates
        #self.Line_EventNo = self.file["Line_Candidates"]["EventNo"].array(library="np")
        #self.Line_SliceNo = self.file["Line_Candidates"]["SliceNo"].array(library="np")
        #self.Line_SpillNo = self.file["Line_Candidates"]["SpillNo"].array(library="np")
        #self.Line_RunNo = self.file["Line_Candidates"]["RunNo"].array(library="np")

## Setup driver to read ND-LAr info

In [7]:
## Call files and load SPINE driver to read spine file

cfg = f'''
base:
  verbosity: warning
build:
  mode: both
  fragments: false
  particles: true
  interactions: true
  units: cm
###  
post:
  match:
    particle:
      match_mode: both
    interaction:
      match_mode: both
###
io:
  reader:
    file_keys: {SPINE_file}
    skip_unknown_attrs: True
    name: hdf5
'''
driver = Driver(yaml.safe_load(cfg))

## Open TMS file and print tree / branch structure

In [8]:
file = uproot.open(TMS_file)

#  Set to track unique tree names
unique_trees = set()

print("Unique top-level trees and their branches:")

# Iterate through items in the ROOT file
for name, obj in file.items():
    # Check if the object is a TTree and if we haven't seen this tree name yet
    if isinstance(obj, uproot.TTree) and name not in unique_trees:
        print(f"\nTree: {name}")
        unique_trees.add(name)  # Add to set to avoid duplicates
        # List branches in the tree
        for branch in obj.keys():
            print(f"  Branch: {branch}")
file.close()

Unique top-level trees and their branches:

Tree: Line_Candidates;2
  Branch: EventNo
  Branch: SliceNo
  Branch: SpillNo
  Branch: RunNo
  Branch: nLinesU
  Branch: nLinesV
  Branch: nLinesX
  Branch: nLinesY
  Branch: nLines3D
  Branch: SlopeU
  Branch: InterceptU
  Branch: Slope_DownstreamU
  Branch: Intercept_DownstreamU
  Branch: Slope_UpstreamU
  Branch: Intercept_UpstreamU
  Branch: SlopeV
  Branch: InterceptV
  Branch: Slope_DownstreamV
  Branch: Intercept_DownstreamV
  Branch: Slope_UpstreamV
  Branch: Intercept_UpstreamV
  Branch: SlopeX
  Branch: InterceptX
  Branch: Slope_DownstreamX
  Branch: Intercept_DownstreamX
  Branch: Slope_UpstreamX
  Branch: Intercept_UpstreamX
  Branch: SlopeY
  Branch: InterceptY
  Branch: Slope_DownstreamY
  Branch: Intercept_DownstreamY
  Branch: Slope_UpstreamY
  Branch: Intercept_UpstreamY
  Branch: DirectionZU
  Branch: DirectionZV
  Branch: DirectionZX
  Branch: DirectionZY
  Branch: DirectionXU
  Branch: DirectionXV
  Branch: DirectionYX
 

## Inspect an ND-LAr spill

In [9]:
spill = 0    # For now only spill 0 works with vertex timing in TMS

data = driver.process(spill)
truth_particles = data[f'truth_particles']
print("Total nof true particles in ND-LAr spill", spill, "/", len(driver), ":", len(truth_particles))

NDLAr_pdg = np.array([p.pdg_code for p in truth_particles])
NDLAr_shape = np.array([p.shape for p in truth_particles])

NDLAr_ancestor_times = np.array([(p.ancestor_t - 1.2e9 * spill) for p in truth_particles])
rounded_NDLAr_ancestor_times = np.round(NDLAr_ancestor_times, 3)

NDLAr_start_times = np.array([(p.t - 1.2e9 * spill) for p in truth_particles])
NDLAr_end_times = np.array([(p.end_t - 1.2e9 * spill) for p in truth_particles])

NDLAr_start_x = np.array([p.position[0] for p in truth_particles])
NDLAr_end_x = np.array([p.end_position[0] for p in truth_particles])

NDLAr_start_y = np.array([p.position[1] for p in truth_particles])
NDLAr_end_y = np.array([p.end_position[1] for p in truth_particles])

NDLAr_start_z = np.array([p.position[2] for p in truth_particles])
NDLAr_end_z = np.array([p.end_position[2] for p in truth_particles])

NDLAr_ke = np.array([p.ke for p in truth_particles])
NDLAr_energy_init = np.array([p.energy_init for p in truth_particles])

NDLAr_truthpart = pd.DataFrame(
    {'PDG': NDLAr_pdg,
     'Shape': NDLAr_shape,
     'Ancestor Time': NDLAr_ancestor_times,
     'Start Time': NDLAr_start_times,
     'End Time': NDLAr_end_times,
     'Start Position X': NDLAr_start_x,
     'End Position X': NDLAr_end_x,
     'Start Position Y': NDLAr_start_y,
     'End Position Y': NDLAr_end_y,
     'Start Position Z': NDLAr_start_z,
     'End Position Z': NDLAr_end_z,
     'KE': NDLAr_ke,
     'Energy_init': NDLAr_energy_init
    })

pd.set_option('display.max_rows', None)
NDLAr_truthpart.head(50)

Total nof true particles in ND-LAr spill 0 / 13 : 770


,PDG,Shape,Ancestor Time,Start Time,End Time,Start Position X,End Position X,Start Position Y,End Position Y,Start Position Z,End Position Z,KE,Energy_init
0,13,1,76.406223,76.406223,230.167359,92.096222,-24.277039,316.716797,-566.628906,-1146.320068,3354.322021,16694.737092,16800.322266
1,11,3,76.406223,146.197139,146.288042,131.072845,132.259827,-89.582108,-89.531952,905.630432,907.025146,8.127530,8.638527
2,11,3,76.406223,145.798188,146.056475,130.839203,129.696289,-87.356232,-89.832764,893.882385,900.376831,46.412285,46.923439
3,11,3,76.406223,143.514508,143.675169,129.502380,128.913055,-74.530380,-75.096069,826.649963,830.597717,12.195357,12.706349
4,11,3,76.406223,142.248762,142.478013,128.788574,127.377777,-67.356506,-69.773590,789.397705,793.558594,16.099968,16.610924
5,11,3,76.406223,140.991196,141.156802,127.996490,126.342499,-60.202057,-59.781906,752.393005,756.330627,14.415388,14.926392
6,11,3,76.406223,139.990745,140.077995,127.317505,126.735046,-54.439819,-55.422394,722.968872,724.325867,6.484902,6.995896
7,11,3,76.406223,139.913936,139.998431,127.264740,127.671997,-53.996872,-53.903824,720.709961,722.685852,7.682404,8.193393
8,11,3,76.406223,136.623036,136.700832,125.133362,125.974457,-35.302887,-35.991989,623.868774,625.395813,5.447755,5.958759
9,11,3,76.406223,134.753315,134.833744,124.230042,124.603394,-24.728760,-26.103577,568.833374,569.120911,5.690864,6.201867


In [10]:
truth_interactions = data[f'truth_interactions']
print("Total nof true interactions in ND-LAr spill", spill, "/", len(driver), ":", len(truth_interactions))

itx = 0
print(f"\nTake a quick look at interaction {itx} / {len(truth_interactions)}:")
print("ID:", truth_interactions[itx].id)
print("Vertex:", truth_interactions[itx].vertex)
print("Vertex Time:", truth_interactions[itx].t)
print("Particle IDs:", truth_interactions[itx].particle_ids)
print("Primary Particle IDs:", truth_interactions[itx].primary_particle_ids)
print("Interaction ID", truth_interactions[itx].interaction_id)
print("Track ID", truth_interactions[itx].track_id)
print("Position:", truth_interactions[itx].position)

NDLAr_vtx_x = np.array([p.vertex[0] for p in truth_interactions])
NDLAr_vtx_y = np.array([p.vertex[1] for p in truth_interactions])
NDLAr_vtx_z = np.array([p.vertex[2] for p in truth_interactions])

NDLAr_interaction_id = np.array([p.interaction_id for p in truth_interactions])
NDLAr_track_id = np.array([p.track_id for p in truth_interactions])
NDLAr_t = np.array([(p.t - 1.2e9 * spill) for p in truth_interactions])

NDLAr_position_x = np.array([p.position[0] for p in truth_interactions])  # Seems to be the same as the 'vertex' variable
NDLAr_position_y = np.array([p.position[1] for p in truth_interactions])
NDLAr_position_z = np.array([p.position[2] for p in truth_interactions])

NDLAr_truthinteraction = pd.DataFrame(
    {'t': NDLAr_t,
     'Vtx x': NDLAr_vtx_x,
     'Vtx y': NDLAr_vtx_y,
     'Vtx z': NDLAr_vtx_z,
     'Interaction id': NDLAr_interaction_id,
     'Track id': NDLAr_track_id
    })

pd.set_option('display.max_rows', None)
NDLAr_truthinteraction.head(85) # 0 - 85

Total nof true interactions in ND-LAr spill 0 / 13 : 86

Take a quick look at interaction 0 / 86:
ID: 0
Vertex: [ 276.66272 -196.11389  808.79236]
Vertex Time: 190.502403
Particle IDs: [11 12 13 14 15 16 17 18 19 20 21 22 23]
Primary Particle IDs: [11 12 13 14 15 16 18]
Interaction ID 24710000000
Track ID 4294967295
Position: [ 276.66272 -196.11389  808.79236]


,t,Vtx x,Vtx y,Vtx z,Interaction id,Track id
0,190.502403,276.662720,-196.113892,808.792358,24710000000,4294967295
1,494.506181,-102.420471,82.577454,636.136780,24710000001,4294967295
2,684.474673,219.380493,108.004028,493.899048,24710000002,4294967295
3,855.082319,-62.927734,-213.308273,473.906158,24710000003,4294967295
4,969.567374,-118.417938,-23.385559,623.098328,24710000004,4294967295
5,1851.062917,8.840057,56.511505,589.128113,24710000006,4294967295
6,2003.030335,191.141846,16.478455,558.594604,24710000007,4294967295
7,2136.083174,-204.364029,18.369858,898.149475,24710000008,4294967295
8,2421.392313,36.647949,-86.342941,512.630920,24710000009,4294967295
9,2953.519672,56.644989,-170.136993,897.176270,24710000011,4294967295


In [11]:
# 1. Group the particle indices by their Start Time
# This creates a Series where the index is 'Time' and the value is a list of row indices
time_to_particle_indices = NDLAr_truthpart.groupby('Ancestor Time').indices

# 2. Map these indices back to the truth interaction dataframe
NDLAr_truthinteraction['particle_indices'] = NDLAr_truthinteraction['t'].map(time_to_particle_indices)

## Open TMS File and look at it

In [12]:
TMS = TMSData(TMS_file)
print("Total Entries:", len(TMS.EventNo))
print("Looking at spill", spill, "/", TMS.SpillNo[-1])
print()

# Create a 'mask' of True/False values to look at a single spill
mask = (TMS.SpillNo == spill)

# Get only the indices in chosen spill (where the mask is True)
TMS_indices = np.where(mask)[0]
print("All TMS indices in spill", spill, ":")
print(TMS_indices)

print()

print("Truth Info Index:", TMS.TruthInfoIndex)  # This just gives the first index of the spill 

Total Entries: 1234
Looking at spill 0 / 12

All TMS indices in spill 0 :
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92]

Truth Info Index: [   0   93  182  280  373  473  566  663  753  849  929 1041 1135]


In [13]:
print("\033[1;4;91mTMS info dump\033[0m")

x_nodes, y_nodes, z_nodes = [], [], []

# Create lists to hold the segmented line data
x_lines_reco, y_lines_reco, z_lines_reco = [], [], []

# Create lists to hold the segmented line data
x_lines_recotrue, y_lines_recotrue, z_lines_recotrue = [], [], []

matched_indices = []
matched_tracks = []
matched_x_nodes, matched_y_nodes, matched_z_nodes = [], [], []
matched_track_start, matched_track_end = [], []

sel_PositionLArStart = []
sel_PositionLArEnd = []
sel_PositionTMSStart = []
sel_PositionTMSEnd = []

print("Truth Spill Times:")
formatted_times = [f"{x:.10f}" for x in TMS.TrueVtxT[spill]]
print(formatted_times[0:50])

truth_info_times = []

print()
print("Truth Info Times")

for i in TMS_indices:  # This gets everything from the current spill from the Truth_Info tree
   truth_info_times.append(float(TMS.NeutrinoX4[i][3]))

print(truth_info_times)

# Now loop over the relevant hits
#for i in TMS_indices:
for i, info_idx in enumerate(TMS_indices, start=0):
    print("\nEntry index", info_idx, "/", max(TMS_indices))
    print("Event", TMS.EventNo[info_idx], "/", max(TMS.EventNo))
    # print("Run No:", TMS.RunNo[i])

    print("Muon Vertex", TMS.MuonVertex[i])
    
    # Store every hit in TMS for this event
    x_nodes.extend(TMS.TrueHitX[i])
    y_nodes.extend(TMS.TrueHitY[i])
    z_nodes.extend(TMS.TrueHitZ[i])

    # Get basic info
    # vertexID = TMS.VertexID[i]
    # trackID = TMS.TrackID[i]
    # PDG = TMS.PDG[i]
   
    # print("TruthInfoIndex:", TruthInfoIndex)
    # print("VertexID:", vertexID)
    # print("TrackID:", trackID)
    # print("PDG:", PDG)

    # Primary Particle info
    RecoTrackPrimaryParticleVtxID = TMS.RecoTrackPrimaryParticleVtxID[i]
    RecoTrackPrimaryParticlePDG = TMS.RecoTrackPrimaryParticlePDG[i]
    RecoTrackPrimaryParticleIndex = TMS.RecoTrackPrimaryParticleIndex[i]

    print("RecoTrackPrimaryParticleVtxID:", RecoTrackPrimaryParticleVtxID)
    print("RecoTrackPrimaryParticlePDG:", RecoTrackPrimaryParticlePDG)    
    print("RecoTrackPrimaryParticleIndex:", RecoTrackPrimaryParticleIndex)

    RecoTrackPrimaryParticleTruePositionEnteringTMS = TMS.RecoTrackPrimaryParticleTruePositionEnteringTMS[i]
    RecoTrackPrimaryParticleTruePositionLeavingTMS = TMS.RecoTrackPrimaryParticleTruePositionLeavingTMS[i]
    RecoTrackPrimaryParticleTruePositionLeavingLAr = TMS.RecoTrackPrimaryParticleTruePositionLeavingLAr[i]

    #Secondary Particle info
    RecoTrackSecondaryParticleIndex = TMS.RecoTrackSecondaryParticleIndex[i]
    RecoTrackSecondaryParticlePDG = TMS.RecoTrackSecondaryParticlePDG[i]
    

    #PositionLArStart.extend(TMS.PositionLArStart[i])
    #PositionLArEnd.extend(TMS.PositionLArEnd[i])
    #PositionTMSStart.extend(TMS.PositionTMSStart[i])
    #PositionTMSEnd.extend(TMS.PositionTMSEnd[i])
    
    # Get track start and end points
    RECO_StartPos = TMS.RECO_StartPos[i]
    RECO_EndPos = TMS.RECO_EndPos[i]

    RECOTRUE_StartPos = TMS.RecoTrackPrimaryParticleTruePositionTrackStart[i]
    RECOTRUE_EndPos = TMS.RecoTrackPrimaryParticleTruePositionTrackEnd[i]

    #BirthPosition = TMS.BirthPosition[i]
    #print("Birth Position:", BirthPosition)
    
    print("RecoTrackPrimaryParticleTruePositionLeavingLAr:", RecoTrackPrimaryParticleTruePositionLeavingLAr)
    print("RecoTrackPrimaryParticleTruePositionEnteringTMS:", RecoTrackPrimaryParticleTruePositionEnteringTMS)
    print("RecoTrackPrimaryParticleTruePositionLeavingTMS:", RecoTrackPrimaryParticleTruePositionLeavingTMS)

    print("RecoTrackSecondaryParticlePDG:", RecoTrackSecondaryParticlePDG)
    print("RecoTrackSecondaryParticleIndex:", RecoTrackSecondaryParticleIndex)

    TrueVtxT = TMS.TrueVtxT[spill][i]
    TrueVtxX = TMS.TrueVtxX[spill][i]
    TrueVtxY = TMS.TrueVtxY[spill][i]
    TrueVtxZ = TMS.TrueVtxZ[spill][i]
    print("\033[91mTrue Vertex Time:", TrueVtxT, "\033[0m")
    print("TrueVtxX:", TrueVtxX)
    print("TrueVtxY:", TrueVtxY)
    print("TrueVtxZ:", TrueVtxZ)

    # Check for matches between TMS tracks and ND-Lar tracks
    mask = np.isclose(rounded_NDLAr_ancestor_times, TrueVtxT, atol=1e-3)

    if np.any(mask):
        matched_tracks.append([i, np.where(mask)[0], TrueVtxT])
    
    if np.any(np.isclose(rounded_NDLAr_ancestor_times, TrueVtxT, atol=1e-3)):
        print("\033[91m")
        print("MATCH")
        matched_indices.append(info_idx)
        start_coord = RECOTRUE_StartPos.flatten()
        end_coord = RECOTRUE_EndPos.flatten()
        print("RECO_StartPos: ", TMS.RECO_StartPos[i])
        print("RECO_EndPos: ", TMS.RECO_EndPos[i])
        print()
        print("RECOTRUE_StartPos:", start_coord)
        print("RECOTRUE_EndPos:", end_coord)
        
        if RECOTRUE_StartPos.size > 0:
            matched_x_nodes.extend([start_coord[0], end_coord[0], None])    # The 'None' lifts the plotly pen
            matched_y_nodes.extend([start_coord[1], end_coord[1], None])
            matched_z_nodes.extend([start_coord[2], end_coord[2], None])
            matched_track_start.append([start_coord[0], start_coord[1], start_coord[2]])
            matched_track_end.append([end_coord[0], end_coord[1], end_coord[2]])
        else:
            print("No reco track associated with this match")
        
        print("\033[0m")
            
    for start, end in zip(RECO_StartPos, RECO_EndPos):
        x_lines_reco.extend([start[0], end[0], None])
        y_lines_reco.extend([start[1], end[1], None])
        z_lines_reco.extend([start[2], end[2], None])

    for start, end in zip(RECOTRUE_StartPos, RECOTRUE_EndPos):
        x_lines_recotrue.extend([start[0], end[0], None])
        y_lines_recotrue.extend([start[1], end[1], None])
        z_lines_recotrue.extend([start[2], end[2], None])

TMS info dump
Truth Spill Times:
['0.7299720049', '19.2659912109', '76.4062271118', '95.9215240479', '152.0167236328', '171.7185211182', '190.5023956299', '190.9931030273', '285.9911804199', '304.3083801270', '361.8605957031', '380.6953735352', '399.3994445801', '399.7138671875', '399.7648620605', '437.8695373535', '456.0197753906', '456.3215026855', '475.6371154785', '494.1216430664', '494.4869079590', '494.5061950684', '513.9501953125', '532.9518432617', '608.8279418945', '627.1353759766', '627.5316162109', '646.9359741211', '665.2269897461', '684.4746704102', '684.5284423828', '684.7940063477', '703.2173461914', '703.6740722656', '722.6079101562', '741.1098022461', '760.3746337891', '760.5453491211', '760.8781127930', '779.3172607422', '779.9150390625', '817.4758300781', '836.6167602539', '836.8333740234', '855.0823364258', '874.0092773438', '893.0070190430', '912.2619018555', '931.5639648438', '969.5673828125']

Truth Info Times
[0.7299720048904419, 76.4062271118164, 304.3083801269

In [14]:
print("Matched TMS indices:", np.array(matched_indices))
#print("Matched Track info:", matched_tracks)

# 1. Convert matched track list to a Pandas DataFrame
matches = pd.DataFrame(matched_tracks, columns=['TMS Truth_Spill ID', 'ND-LAr Matched Particles', 'Time'])

# 2. Clean up the 'Matched Track' column 
# (Turns 'array([1, 2])' into '[1, 2]' for better readability)
matches['ND-LAr Matched Particles'] = matches['ND-LAr Matched Particles'].apply(lambda x: list(x))

# 3. Display the first 50 rows
matches.head(50)

Matched TMS indices: [ 2  6  8 15 21 24 28 29 40 42 44 48 49 67 69 72 74 83 84 92]


,TMS Truth_Spill ID,ND-LAr Matched Particles,Time
0,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",76.406227
1,6,"[11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 2...",190.502396
2,8,[24],285.991180
3,15,[25],437.869537
4,21,"[26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 3...",494.506195
5,24,"[51, 52, 53, 54]",608.827942
6,28,"[55, 56, 57, 58, 59]",665.226990
7,29,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71]",684.474670
8,40,"[72, 73, 74, 75]",779.915039
9,42,"[76, 77, 78, 79]",836.616760


# Setup Figure

In [15]:
def setup_figure(figure, NDLAr_idx):
    figure.update_layout(
        width=1600,
        height=900,
        scene=dict(
            xaxis=dict(title="x [mm]", range=[-4500, 4500]),
            yaxis=dict(title="y [mm]", range=[-4000, 1000]),
            zaxis=dict(title="z [mm]", range=[0, 20000]),
            camera=dict(
                eye=dict(x=2.0, y=4.0, z=0.0),
                up=dict(x=0, y=-1, z=0)           # define Y as "up"
            ),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=4)
        ),
        
        margin=dict(l=0, r=0, b=0, t=50),
        
        showlegend=True,
        legend=dict(
            x=0.02,          # Places legend 2% from the left edge
            y=0.98,          # Places legend 2% from the top edge
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.8)', # Semi-transparent white background
            bordercolor='Black',
            borderwidth=1,
            itemsizing='constant'
        ),

        title={
            'text': f"ND-LAr Interaction {NDLAr_idx}",
            'x': 0.5,            # Centers the title
            'xanchor': 'center'
        }
    )
    
    figure.data = []  # Clear all traces
    
    plot_box(figure, -3730, 3730, -3702.23, 997.77, 11177.5, 18502.5, "blue", "TMS detector")
    plot_box(figure, -3478.48, 3478.48, -2166.71, 829.282, 4179.24, 9135.88, "black", "ND-LAr detector")

## Plot ND-LAr event by id

In [16]:
def plot_NDLAr_event(figure, df_parts, df_vtx, indices):
    for idx in indices:
        # 1. Get the target time
        target_time = df_parts.iloc[idx]['Ancestor Time']
        
        # 2. Filter dataframes
        group_parts = df_parts[df_parts['Ancestor Time'] == target_time]
        group_vtx = df_vtx[abs(df_vtx['t'] - target_time) < 1e-5]
    
        # --- Add ALL Tracks (Converting cm -> mm) ---
        x_coords, y_coords, z_coords = [], [], []
        for _, row in group_parts.iterrows():
            # Multiply each coordinate by 10
            x_coords.extend([row['Start Position X']*10, row['End Position X']*10, None])
            y_coords.extend([row['Start Position Y']*10, row['End Position Y']*10, None])
            z_coords.extend([row['Start Position Z']*10, row['End Position Z']*10, None])

    figure.add_trace(
        go.Scatter3d(
            x=x_coords, y=y_coords, z=z_coords,
            mode='lines',
            line=dict(color='darkred', width=3),      # '#e6ab02'
            name=f"ND-LAr Tracks (t={target_time:.3f})",
            hoverlabel_namelength=-1 
        )
    )

    for idx in indices:
        figure.add_trace(
            go.Scatter3d(
                x=truth_interactions[idx].points[:, 0]*10,
                y=truth_interactions[idx].points[:, 1]*10,
                z=truth_interactions[idx].points[:, 2]*10,
                mode="markers",
                marker=dict(size=3, color='#1b9e77'),
                name=f"ND-LAr interaction id {idx}"
            )
        )

        figure.add_trace(
            go.Scatter3d(
                x=truth_particles[idx].points[:, 0]*10,
                y=truth_particles[idx].points[:, 1]*10,
                z=truth_particles[idx].points[:, 2]*10,
                mode="markers",
                marker=dict(size=3, color='black'),
                name=f"ND-LAr particle id {idx}"
            )
        )

    # --- Add ALL Vertices (Converting cm -> mm) ---
    if not group_vtx.empty:
        figure.add_trace(go.Scatter3d(
            # Multiply entire series by 10
            x=group_vtx['Vtx x'] * 10,
            y=group_vtx['Vtx y'] * 10,
            z=group_vtx['Vtx z'] * 10,
            mode='markers',
            marker=dict(size=3, color='red', symbol='x'),
            name=f"True Primary Vertex ND-LAr (t={target_time:.3f})",
            hovertext=[f"ID: {i_id}" for i_id in group_vtx['Interaction id']],
            hoverinfo='text'
        ))

    # --- Layout Update (Units changed to mm) ---
    figure.update_layout(
        title_text=f"Matched Vertex/Ancestor Time: t = {target_time:.3f} &mu;s",
        title_x=0.5,
        margin=dict(t=80, b=20, l=20, r=20),
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)'
        ),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    return target_time

In [17]:
def plot_NDLAr_interaction(figure, idx):
    vtx_x = truth_interactions[idx].vertex[0] * 10
    vtx_y = truth_interactions[idx].vertex[1] * 10
    vtx_z = truth_interactions[idx].vertex[2] * 10
    vtx_t = truth_interactions[idx].t
 
    figure.add_trace(
        go.Scatter3d(
            x=truth_interactions[idx].points[:, 0]*10,
            y=truth_interactions[idx].points[:, 1]*10,
            z=truth_interactions[idx].points[:, 2]*10,
            mode="markers",
            marker=dict(size=2, 
                    color=NDLAr_colour,
                    symbol='circle'
            ),
            name=f"(#{idx}) ND-LAr Hits"
        )
    )

    figure.add_trace(
        go.Scatter3d(
            x = [vtx_x],
            y = [vtx_y],
            z = [vtx_z],
            mode="markers",
            marker=dict(size=4, 
                    color=NDLAr_colour,
                    symbol='square-open'
            ),
            name=f"(#{idx}) ND-LAr Vertex [{vtx_x:.3f}, {vtx_y:.3f}, {vtx_z:.3f}, {vtx_t:.3f}]"
        )
    )    

## Plot TMS matches

In [18]:
def plot_TMS_matches(figure, truth_info_idx, truth_spill_idx):
    if truth_info_idx != -1:
        for i, info_idx in enumerate(truth_info_idx, start=0):
            info_time = TMS.NeutrinoX4[info_idx][3]
            #print("TMS Truth_Info time: ", info_time)
    
            neutrino_x, neutrino_y, neutrino_z = [TMS.NeutrinoX4[info_idx][0]], [TMS.NeutrinoX4[info_idx][1]], [TMS.NeutrinoX4[info_idx][2]]
    
            figure.add_trace(go.Scatter3d(
                x=neutrino_x, y=neutrino_y, z=neutrino_z,
                mode='markers',
                marker=dict(
                    size=5,
                    color=strong_colours[i],
                    symbol='circle-open'
                ),
                #name=f'TMS True Hits (t={target_time:.3f})'
                name=f'(#{info_idx}) TMS Truth_Info Vertex [{TMS.NeutrinoX4[info_idx][0]:.3f}, {TMS.NeutrinoX4[info_idx][1]:.3f}, {TMS.NeutrinoX4[info_idx][2]:.3f}, {info_time:.3f}]',
                hoverlabel_namelength=-1 
            ))
            
            hits_x = (TMS.TrueHitX[info_idx])
            hits_y = (TMS.TrueHitY[info_idx])    
            hits_z = (TMS.TrueHitZ[info_idx])
        
            figure.add_trace(go.Scatter3d(
                x=hits_x, y=hits_y, z=hits_z,
                mode='markers',
                marker=dict(
                    size=2,
                    color=light_colours[i],
                    symbol='circle'
                ),
                #name=f'TMS True Hits (t={target_time:.3f})'
                name=f'(#{info_idx}) TMS Truth_Info Hits',
                hoverlabel_namelength=-1 
            ))

    if truth_spill_idx != -1:
        for j, spill_idx in enumerate(truth_spill_idx, start=0):
            spill_time = TMS.TrueVtxT[spill][spill_idx]
            #print("TMS Truth_Spill time:", spill_time)
    
            col = 'black'
            
            if truth_info_idx != -1:
                for m, time_match in enumerate(truth_info_idx, start=0):
                    if abs(spill_time - TMS.NeutrinoX4[truth_info_idx[m]][3]) < 1e-3:
                        col = strong_colours[m]
            
            figure.add_trace(go.Scatter3d(
                x=[TMS.TrueVtxX[spill][spill_idx]], y=[TMS.TrueVtxY[spill][spill_idx]], z=[TMS.TrueVtxZ[spill][spill_idx]],
                mode='markers',
                marker=dict(
                    size=5,
                    color=col,
                    symbol='cross',
                ),
                #name=f'TMS True Hits (t={target_time:.3f})'
                name=f'(#{spill_idx}) TMS Truth_Spill Vertex [{TMS.TrueVtxX[spill][spill_idx]:.3f}, {TMS.TrueVtxY[spill][spill_idx]:.3f}, {TMS.TrueVtxZ[spill][spill_idx]:.3f}, {spill_time:.3f}]',
                hoverlabel_namelength=-1 
            ))

    return figure

## Mapping between ND-LAr and the two TMS truth trees

In [19]:
def get_TMS_indices(df, NDLAr_idx): # Filter the dataframe where NDLAr_idx matches
    row = df[df['NDLAr_idx'] == NDLAr_idx]

    if not row.empty:
        # Access the values (using .iloc[0] to get the first match found)
        info_idx = row['Info_idx'].iloc[0]
        spill_idx = row['Spill_idx'].iloc[0]
        return info_idx, spill_idx
    else:
        return None, None

In [20]:
print("Nof ND-LAr truth interactions:", len(truth_interactions))
print("Nof TMS Truth Info events:", len(TMS.EventNo))
print("Nof TMS Truth Spill events:", len(TMS.TrueVtxT[spill]))

#print(TMS.EventNo)
#print(TMS.EventNo_TruthSpill)
#print(TMS_indices)

matched_data = []
time_window = 1e-3

for NDLAr_idx in range(len(truth_interactions)):
    NDLAr_time = truth_interactions[NDLAr_idx].t - (1.2e9 * spill)
    NDLAr_itx_id = truth_interactions[NDLAr_idx].interaction_id
    
    TMS_info_match = []
    TMS_info_idx_match = []
    TMS_info_prim_pid = []
    TMS_info_sec_pid = []

    TMS_spill_match = []
    TMS_spill_idx_match = []

    for TMS_info_idx in TMS_indices:
        #if NDLAr_idx == 0:
            #print(TMS.NeutrinoX4[TMS_info_idx][3])
        if abs(TMS.NeutrinoX4[TMS_info_idx][3] - NDLAr_time) < time_window:
            TMS_info_match.append(TMS.NeutrinoX4[TMS_info_idx][3])
            TMS_info_idx_match.append(TMS_info_idx)
            TMS_info_prim_pid.append(TMS.RecoTrackPrimaryParticleIndex[TMS_info_idx])
            TMS_info_sec_pid.append(TMS.RecoTrackSecondaryParticleIndex[TMS_info_idx])
    for TMS_spill_idx in range(len(TMS.TrueVtxT[spill])):
        if abs(TMS.TrueVtxT[spill][TMS_spill_idx] - NDLAr_time) < time_window:
            TMS_spill_match.append(TMS.TrueVtxT[spill][TMS_spill_idx])
            TMS_spill_idx_match.append(TMS_spill_idx)
            #print(f"NDLAr: {NDLAr_idx}, TMS_Spill: {TMS_spill_match_idx}")
    
    # Store results in a dictionary
    entry = {
        "NDLAr_idx": NDLAr_idx,
        "NDLAr_time": NDLAr_time,
        "NDLAr_interaction_id": NDLAr_itx_id,
        "Info_idx": TMS_info_idx_match if TMS_info_match else -1,    # Store -1 if no match was found
        "Info_time": TMS_info_match if TMS_info_match else -1,
        "Info_prim_pid": TMS_info_prim_pid if TMS_info_match else -1,
        "Info_sec_pid": TMS_info_sec_pid if TMS_info_match else -1,
        "Spill_idx": TMS_spill_idx_match if TMS_spill_match else -1,
        "Spill_time": TMS_spill_match if TMS_spill_match else -1,
    }
    matched_data.append(entry)

# Create the DataFrame
matches = pd.DataFrame(matched_data)
#matches['Info_idx'] = matches['Info_idx'].astype("Int64")
#matches['Spill_idx'] = matches['Spill_idx'].astype("Int64")

pd.set_option('display.max_rows', None)
matches

Nof ND-LAr truth interactions: 86
Nof TMS Truth Info events: 1234
Nof TMS Truth Spill events: 435


,NDLAr_idx,NDLAr_time,NDLAr_interaction_id,Info_idx,Info_time,Info_prim_pid,Info_sec_pid,Spill_idx,Spill_time
0,0,190.502403,24710000000,-1,-1,-1,-1,[6],[190.5024]
1,1,494.506181,24710000001,-1,-1,-1,-1,[21],[494.5062]
2,2,684.474673,24710000002,-1,-1,-1,-1,[29],[684.4747]
3,3,855.082319,24710000003,-1,-1,-1,-1,[44],[855.08234]
4,4,969.567374,24710000004,-1,-1,-1,-1,[49],[969.5674]
5,5,1851.062917,24710000006,-1,-1,-1,-1,[83],[1851.0629]
6,6,2003.030335,24710000007,-1,-1,-1,-1,[94],[2003.0304]
7,7,2136.083174,24710000008,-1,-1,-1,-1,[98],[2136.0833]
8,8,2421.392313,24710000009,-1,-1,-1,-1,[114],[2421.3923]
9,9,2953.519672,24710000011,[23],[2953.5198],[[]],[[]],[134],[2953.5198]


## Draw Figure

In [21]:
@interact(NDLAr_idx=(0, len(truth_interactions)-1))
def update_plot(NDLAr_idx):
    fig = go.Figure()

    setup_figure(fig, NDLAr_idx)
    plot_NDLAr_interaction(fig, NDLAr_idx)
    
    idx_info, idx_spill = get_TMS_indices(matches, NDLAr_idx)
    plot_TMS_matches(fig, idx_info, idx_spill)
    
    fig.show()

interactive(children=(IntSlider(value=42, description='NDLAr_idx', max=85), Output()), _dom_classes=('widget-i…